# LABORATORIO 9: DATOS NO ESTRUCTURADOS Y FUNCIONES CORTEX LLM

👉 En esta lección, revisaremos los documentos PDF que se proporcionaron y usaremos una función Cortex de Snowflake para analizar estos documentos, y escribiremos la información en una tabla. Se empleará una segunda función Cortex para etiquetar (o clasificar) las filas de datos en la tabla. Construiremos una consulta analítica y la visualizaremos usando Streamlit para mostrar el desglose de la información entre las categorías. Además, finalmente haremos un recorrido rápido por las capacidades de tres otras funciones Cortex. 

Para comenzar, reunamos la **información de contexto** que usaremos durante este laboratorio. 

- Haga clic en el botón **Start** para activar este cuaderno.

- Ejecute la siguiente celda Python.


#### :warning: Debe volver a ejecutar la celda a continuación siempre que se inicie una nueva sesión para este cuaderno para configurar “variables” para usar en celdas posteriores. :warning:


In [ ]:
import streamlit as st
from snowflake.snowpark.context import get_active_session
session = get_active_session()
user = session.get_current_user().strip('"')
session.use_database(f'{user}_GARDEN_PLANTS')
session.use_schema('VEGGIES')
print('Su información de contexto actual:')
print('---------------------------------')
print(session)
print('Su usuario actual es' + user)

## Trabaje con datos no estructurados en Snowflake 📓

En secciones anteriores de este curso, analizamos los datos estructurados (filas y columnas) y datos semiestructurados (formato JSON). Snowflake también incluye características y funciones para trabajar con [**datos no estructurados**](https://docs.snowflake.com/en/user-guide/unstructured-intro), que los analistas estiman que ahora representan alrededor del 80 % de los datos del mundo.

Entones, ¿qué es exactamente?

Los **datos no estructurados** son información que no se adecua a un modelo o esquema de datos predefinidos. Por lo general, los datos no estructurados contienen mucho texto, como respuestas a formularios y conversaciones en redes sociales, pero también comprenden imágenes, video y audio. En esta categoría, están incluidos los tipos de archivo específicos de un sector, como VCF (genómico), KDF (semiconductores) o HDF5 (aeronáutica).

Snowflake AI Data Cloud puede ayudar a acceder a los archivos de **datos no estructurados**, además compartirlos y procesarlos.


## Un escenario de datos no estructurados 📓

Solo imagine por un momento que, con el tiempo, ha reunido docenas de datos interesantes y detalles sobre las plantas que disfruta cuidar, y que exportó estos “fragmentos” de información como archivos PDF en su laptop. ¿No sería bueno encontrar una forma de llevar esos datos a Snowflake para usarlos junto con la información que ya acumuló?

Este es precisamente el escenario en el que trabajaremos en los ejercicios de este laboratorio. Usted deberá hacer lo siguiente:
1. Analizar la información de archivos PDF preconfigurados usando una función Cortex LLM.
1. Extraer e ingerir los datos en una nueva tabla de Snowflake.
1. Explorar las funciones Cortex LLM de Snowflake para analizar y generar usando estos datos. :robot_face:


### Produzca una lista de los archivos PDF suministrados 🥋

Una colección de aproximadamente 130 archivos PDF que incluyen datos y detalles interesantes sobre plantas se cargó en la fase **course_files** de **common_db.resources** para usted por el equipo de Servicios de Educación.

**Modifique** el siguiente extracto de código SQL que se encuentra a continuación y **ejecute la celda** para `LIST`(enlistar) los archivos PDF suministrados en la fase designada:
- Reemplace los caracteres hash `(#)` por el comando y la sintaxis adecuados para crear la instrucción SQL correcta.
- Ejecute la celda SQL.


In [ ]:
LIST @common_db.resources.course_files/garden_kb;

## Tablas de directorio 📓

Al trabajar con datos no estructurados, las **tablas de directorio** son realmente útiles y de muchas formas superiores al trabajo con el comando `LIST`.

Una [tabla de directorio](https://docs.snowflake.com/en/user-guide/data-load-dirtables) es un objeto implícito montado en una fase (no un objeto de base de datos separado) y es conceptualmente similar a una tabla externa porque almacena metadatos al nivel de archivo sobre los archivos de datos en la fase. Se admiten fases internas y externas. Lo bueno es que estos objetos nos permiten consultar el contenido de un directorio como si fuera una tabla. El proceso de crear una tabla de directorio es tan simple como habilitar la opción en el momento de crear una fase o modificar una fase después de crearla. 

- De hecho, usted creó una tabla de directorio en el Laboratorio 8, a pesar de que quizás ni siquiera se dio cuenta, dado que esta es una opción predeterminada incluida cuando usa el asistente de Snowsight para crear un objeto de fase.


Puede acceder al resultado de la tabla de directorio con una función de tabla y derivar nombres y ubicaciones de archivos con facilidad a varias funciones Snowflake para su procesamiento..


### Consulte una tabla de directorio 🥋

Eche un vistazo a la estructura de la siguiente consulta:
- Es una instrucción `SELECT` habitual.
- Tome nota del uso de la palabra clave **DIRECTORY** para acceder a la función de tabla.
- En este caso, podemos elegir columnas para devolver `*` (“todas”).
- También podemos filtrar nuestros resultados (como lo hacemos para solo devolver aquellos archivos en el subdirectorio “garden_kb” de esta fase).

Ejecute esta consulta y revise el resultado, prestando atención los tipos de información que devuelve sobre los archivos incluidos en esta ubicación de fase.


In [ ]:
SELECT *    
FROM DIRECTORY('@common_db.resources.course_files')
WHERE CONTAINS(relative_path, 'garden_kb/');

## Revise los archivos PDF suministrados 🥋

### ¿Qué información sobre jardinería se recopiló?

Como se mencionó antes, se ha cargado una colección de aproximadamente 130 archivos PDF que contiene datos y detalles interesantes sobre plantas para usted por el equipo de Servicios de Educación.

- La colección de archivos PDF se llaman de **snippet_1.pdf** a **snippet_129.pdf**. 

Podría resultarle interesante saber el tipo de información que contienen estos archivos “factoid”.

Aquí se muestra un ejemplo de esta colección de archivos PDF sobre conocimientos de plantas y jardinería. El texto proviene del primer archivo en la colección y se relaciona con **alcachofas**.

¿Cómo cultivar **alcachofas**? ¡Algunos buenos consejos por aquí!

![Muestra de fragmento (imagen)](https://edu-cdev-images.s3.us-west-2.amazonaws.com/ob/ob_artichokes_1.png)


### Descargue uno de los archivos PDF “factoid” de plantas y revíselo 📓

Arriba pudo ver un ejemplo del contenido en un archivo PDF de nuestra colección de conocimientos sobre plantas y jardinería. Pero descarguemos y abramos otro archivo PDF y revisemos.

- En el siguiente ejemplo, analizaremos el archivo: **snippet_9.pdf**.

- El código SQL (integrado en Python) en este ejemplo usa una función de Snowflake que genera enlaces a archivos almacenados en fases a los que incluso puede acceder “fuera” de Snowflake. Esto resulta útil al trabajar con datos **no estructurados**. Consulte la documentación de Snowflake para obtener más información sobre la función [GET_PRESIGNED_URL](https://docs.snowflake.com/en/sql-reference/functions/get_presigned_url).


### Ejecute la siguiente celda de código Python y use el enlace generado para descargar el archivo 🥋

**Haga clic con el botón derecho** en el enlace generado y en **Open** en la nueva pestaña o ventana del navegador (hacer clic en el vínculo NO funcionará).

**Abra** el archivo en una aplicación para ver PDF de su máquina local si no se abrió automáticamente en el navegador.

- Veamos la información que se incluye en este archivo PDF acerca de su verdura favorita: ¡espárragos! :leafy_green:

- No dude en analizar algunos de los demás archivos, numerados del 1 al 129, después de trabajar en el primer ejemplo.


In [ ]:
snowpark_df = session.sql("SELECT GET_PRESIGNED_URL(@common_db.resources.course_files, 'garden_kb/snippet_9.pdf')")
collected_data = snowpark_df.collect()
st.write('Abre el siguiente enlace en una nueva pestaña o ventana del navegador y revisa')
st.write(collected_data[0][0])
st.write('Espárragos. Quién diría!!!')

## Presentación de Cortex AI de Snowflake📓

AI Data Cloud de Snowflake incluye una serie de características y funciones que le dan acceso al instante a modelos de lenguaje grandes (LLM) líderes en la industria que fueron entrenados por investigadores en compañías como Mistral, Reka, Meta y Google, incluida Snowflake Arctic, un modelo empresarial abierto desarrollado por Snowflake.

Dado que estos LLM están completamente alojados y administrados por Snowflake, su uso **NO** requiere configuración. Los datos permanecen en Snowflake, lo que le permite tener el rendimiento, la escalabilidad y el gobierno que espera.

💡 Consulte la documentación de Snowflake para conocer [el estado de la versión y la disponibilidad](https://docs.snowflake.com/en/guides-overview-ai-features) de las siguientes funciones.

![Snowflake para IA generativa (imagen)](https://edu-cdev-images.s3.us-west-2.amazonaws.com/ob/ob_gen_ai_1_v2.png)

### Funciones de Cortex LLM de Snowflake

Un subconjunto de estas opciones de IA de Snowflake disponible es **Funciones de Cortex LLM**. Estas se proporcionan como funciones SQL y también están disponibles en Python, lo que facilita el acceso a sus potentes capacidades.

Las funciones de Cortex LLM pueden agruparse en las siguientes categorías:

- Funciones específicas de las tareas

- Funciones de ayuda

- Función `COMPLETE`

👉 Trabajaremos en algunas de estas funciones para ingerir y “procesar” nuestros datos sobre plantas y jardinería en el resto de este laboratorio.


## Utilice `PARSE_DOCUMENT()` para extraer texto 📓

Cortex [parse_document()](https://docs.snowflake.com/user-guide/snowflake-cortex/parse-document) es una función específica de tareas de Cortex LLM que proporciona la capacidad para extraer texto o diseño de documentos almacenados en una fase interna o externa. 

Es una función SQL. Dado que está completamente alojada y administrada por Snowflake, su uso no requiere configuración. Esto significa que basta con dirigir la función `PARSE_DOCUMENT` a una fase en la que se almacenan documentos PDF para extraer datos de texto o de diseño. En pocas palabras, todo lo que se requiere es lo siguiente:

- El nombre de una fase de la que leer.

- El documento PDF dentro de esa fase de la que quiere extraer texto (en este punto solo se admiten archivos PDF).

- Puede leer diseños de documentos, pero seleccionaremos el modo **OCR** para trabajar, que es excelente para manejar la extracción de texto.

💡 **Consejo**: para casos de uso de extracción de documentos más sofisticados, se recomienda que consulte el modo **LAYOUT** de esta función o que revise el servicio [Document AI](https://docs.snowflake.com/en/user-guide/snowflake-cortex/document-ai/overview) de Snowflake.


### Ejemplo de extracción de texto 🥋

Revise la siguiente instrucción SQL:

- Usa una referencia totalmente calificada a la función `PARSE_DOCUMENT()`.

- Proporcionamos una referencia a la fase de documentos PDF y un archivo individual dentro de ésta (este es el ejemplo “Espárragos” que abrió antes).

Es así de simple. Todo lo que hace es llamar esta función SQL “integrada” y, en segundo plano, Snowflake abrirá y leerá el contenido del archivo y lo devolverá.

Continúe y ejecute el siguiente código:


In [ ]:
SELECT SNOWFLAKE.CORTEX.PARSE_DOCUMENT (
        @common_db.resources.course_files,
        'garden_kb/snippet_9.pdf',
        {'mode': 'OCR'}
    ) AS output;

### Revise el resultado `PARSE_DOCUMENT()` 🥋

Tenga en cuenta que el resultado de esta función se proporciona en un formato de datos semiestructurados:

![Resultado de Parse_document() (imagen)](https://edu-cdev-images.s3.us-west-2.amazonaws.com/ob/ob_parse_document_1.png)

Hay campos **content** y **metadata**. 

💡 **Consejo**: Recuerde que, tal como aprendió en el **Laboratorio 8**, podemos desglosar los campos anidados incluidos en estas estructuras de datos semiestructurados mediante el operador `:`.


## Cree un flujo de trabajo de extracción de texto 📓

Ahora que tenemos un método para extraer el texto de los documentos PDF sobre plantas y jardinería suministrados, queremos llevarlos a Snowflake para que podamos usarlos.

👉 Queremos crear una **base de conocimientos** a partir de toda la información sobre plantas y jardinería que hemos recopilado, y podemos ver muchos usos posibles para estos datos.

Aquí está el flujo de trabajo:

- La información de nuestra base de conocimientos comienza como datos **NO ESTRUCTURADOS** en archivos PDF.

- La función `PARSE_DOCUMENT()` de Cortex se ejecuta para extraer el texto de los archivos PDF y a un formato de datos **SEMIESTRUCTURADOS**.

- Con la sintaxis de Snowflake, podemos analizar el resultado de la función Cortex para devolver contenido de datos **ESTRUCTURADOS**.


### Cree una tabla para retener los datos de la base de conocimientos 🥋 

Primero, necesitamos crear una tabla para retener la información de referencia extraída de los documentos PDF.

Continúe y ejecute el siguiente código para hacerlo en el esquema **(animal)_GARDEN_PLANTS.VEGGIES**. Tenga en cuenta que incluiremos una columna para el nombre del paciente, a la que se refiere el contenido del archivo PDF. 


In [ ]:
CREATE OR REPLACE TABLE vegetable_knowledge_base (
    source_document STRING, -- el nombre del documento
    insight STRING,         -- el "factoid" contenido en el archivo
    plant_name STRING       -- el nombre de la planta que hace referencia el "factoid"
);

### Extraiga luego los datos `INSERT` a la nueva tabla 🥋

Comencemos por mencionar algunas de las características y funciones que hemos aprendido juntos en este laboratorio.

La siguiente instrucción `INSERT` utiliza las **tablas de directorio** y la función `PARSE_DOCUMENT()`. 

- La lista de la **tabla de directorio** devuelve el nombre de cada uno de nuestros archivos PDF en el subdirectorio **garden_kb** de la fase **common_db.resources.course_files**.

- La ubicación de cada documento PDF se pasa a la función `PARSE_DOCUMENT()` para que se extraiga el texto.

- El texto extraído se devuelve en un formato semiestructurado y se aísla el elemento **content** y se funde en un `STRING` con: `:content::STRING as extract`.

**Ejecute** el código que se encuentra debajo para escribir el nombre de archivo y el texto extraído para cada archivo PDF en su nueva tabla.


In [ ]:
INSERT INTO vegetable_knowledge_base (source_document, insight)
    SELECT 
        split_part(relative_path,'/',-1) as file_name, 
        SNOWFLAKE.CORTEX.PARSE_DOCUMENT (
            @common_db.resources.course_files,
            relative_path,
            {'mode': 'OCR'}
        ):content::STRING as extract    
    from directory('@common_db.resources.course_files')
    where contains(relative_path, 'garden_kb/')
;

### Compruebe su trabajo 🎯

Se deben haber insertado **129** filas en su nueva tabla. Ahora echemos un vistazo a la “forma” de estos datos en la tabla.

- **Vuelva a escribir** el siguiente fragmento de consulta para devolver **todas** las filas de su nueva tabla de base de conocimientos.


In [ ]:
SELECT *
FROM vegetable_knowledge_base;

### ¡Tenemos un problema!

Hemos logrado extraer los datos de los archivos PDF suministrados y los escribimos en una tabla, pero aquí hay un problema…

Sin revisar el contenido de la columna **INSIGHT**, no podemos saber con qué planta se relaciona cada una de las filas de información. Todas las columnas **PLANT_NAME** están vacías. Este es un problema al implementar una base de conocimientos.

_Podríamos_ leer cada fila y actualizar la columna **PLANT_NAME** de forma manual, pero esto llevará mucho tiempo con las 129 filas que contiene esta tabla, y es imposible de lograr con un conjunto de datos de tamaño mucho mayor.

![Tabla de base de conocimientos 1 (imagen)](https://edu-cdev-images.s3.us-west-2.amazonaws.com/ob/ob_kb_table_query_1.png)


## Use `CLASSIFY_TEXT()` en el texto que ha extraído 📓

Afortunadamente, Snowflake Cortex incluye una función LLM llamada [CLASSIFY_TEXT()](https://docs.snowflake.com/en/sql-reference/functions/classify_text-snowflake-cortex). Como lo sugiere su nombre, esto clasifica datos de texto libre de forma que usted proporciona.

- Es fácil de invocar desde SQL y, al igual que las demás funciones Cortex LLM, no requiere ninguna configuración: se incluye como parte de la oferta Snowflake.

- La función devuelve una cadena que incluye un objeto JSON. El objeto JSON incluye la categoría según la que se clasificó la solicitud de entrada. Si se dan argumentos inválidos, se devuelve un error.

![Clasifique el uso del texto (imagen)](https://edu-cdev-images.s3.us-west-2.amazonaws.com/ob/ob_classify_text_1.png)


### Ejemplo de clasificación 🥋

Revise el siguiente ejemplo que se relaciona con nuestro caso de uso. Estamos analizando la cadena de entrada **manzana (apple)** a la función de Cortex LLM para revisar y, con suerte, categorizar correctamente como **fruit**, **veggie** o **flower**.

Continúe y ejecute el siguiente código.

- ¿Fue preciso?

- Pruebe con **tomate** como entrada. :grinning:


In [ ]:
SELECT SNOWFLAKE.CORTEX.CLASSIFY_TEXT('apple', ['fruit', 'veggies', 'flowers']);

### Un caso de uso de clasificación expandida 📓

¡Esto es GENIAL! Esto significa que no tenemos que actualizar manualmente las 129 filas en nuestra tabla, podemos dejar que Snowflake haga el trabajo pesado.

- Necesitamos escribir una instrucción `UPDATE`. 

- Para cada fila de la tabla, analizamos la columna **INSIGHT** como entrada.

- Podemos construir una lista de nombres de todas las plantas conocidas a las que almacenamos referencias de la tabla **VEGETABLE_DETAILS**, que se creó y cargó en un laboratorio anterior.

- Podemos desglosar en resultados de datos semiestructurados para extraer el valor de la etiqueta.


### Ejecute la clasificación 🥋

Revise la siguiente instrucción SQL:

- Usa una referencia totalmente calificada a la función `CLASSIFY_TEXT()`.

 - La sintaxis en la **LINEA 4** posiblemente sea nueva para usted, pero lo que todo esto hace es crear un `ARRAY` de nombres de plantas de la tabla **vegetable_details** para analizar la función de Cortex LLM.
    - Esto ahorra tiempo, ya que no tiene que escribir `Artichoke...Zucchini` manualmente.

Continúe y ejecute el siguiente código:


In [ ]:
UPDATE vegetable_knowledge_base
SET plant_name = SNOWFLAKE.CORTEX.CLASSIFY_TEXT(
    insight, 
    (SELECT ARRAY_AGG(plant_name) WITHIN GROUP (ORDER BY plant_name ASC) FROM vegetable_details) -- ENSAMBLE DE CATEGORÍAS
):label::STRING
WHERE insight IS NOT NULL 
AND insight <> '';

### Compruebe su trabajo 🥋

Las **129** filas deben haberse **actualizado** en la tabla de su base de conocimientos. Ahora eche un vistazo a los datos en la tabla.

- Ejecute la siguiente consulta para devolver todas las filas de la tabla de la base de conocimientos.

- Confirme que la columna **PLANT_NAME** ahora está completa para **TODAS** las filas.


In [ ]:
SELECT *
FROM vegetable_knowledge_base;

## Analice los datos de su base de conocimientos 📓

Con la tabla de la base de conocimientos cargada de los archivos PDF suministrados y cada fila categorizada según el nombre de la planta, sería útil comprender la distribución de estos datos.

- ¿Cuántos “factoids” tenemos por planta? 

- ¿Tenemos más información sobre algunas plantas que otras?

- ¿Nos faltan datos para algunas de nuestra colección de plantas?

Resulta útil saber todo esto para ayudarnos a priorizar nuestros esfuerzos por ampliar esta base de conocimientos.


### Ejecute una consulta analítica 🥋

La siguiente consulta es un ejemplo de uno de los tipos de operaciones [JOIN](https://docs.snowflake.com/en/sql-reference/constructs/join) que admite Snowflake.

En esto, queremos obtener la lista completa de nombres de plantas de la tabla **vegetable_details**. Para cada planta, queremos elaborar un recuento del número de filas en la tabla **vegetable_knowledge_base**. Usar `LEFT OUTER JOIN` significa que, incluso si no hay filas para una planta en particular en **vegetable_knowledge_base**, todavía podemos asignar un valor de cero a esta, en lugar de omitirla por completo de los resultados.

Ejecute y revise la siguiente consulta para comprender el recuento de artículos por planta.

💡 **Consejo**:  Asignaremos un nombre a esta celda, **knowledge_base_analytical_query**, para hacer referencia de manera sencilla a su resultado más adelante.


In [ ]:
SELECT a.plant_name, 
       nvl(count(b.*),0) AS kb_article_count
FROM vegetable_details a
LEFT OUTER JOIN vegetable_knowledge_base b
ON a.plant_name = b.plant_name
GROUP BY a.plant_name
ORDER BY 1;

## Streamlit en Snowflake 📓

[Streamlit](https://streamlit.io/) es una biblioteca de Python de código abierto que permite crear y compartir con facilidad aplicaciones web personalizadas para machine learning y data science.

[Streamlit en Snowflake](https://docs.snowflake.com/en/developer-guide/streamlit/about-streamlit) es una implementación de esta tecnología en Snowflake AI Data Cloud. 

- Snowflake administra el cómputo y el almacenamiento subyacentes para aplicaciones de Streamlit.

- Las aplicaciones de Streamlit son objetos de Streamlit y usan el control de acceso basado en roles (RBAC) para administrar el acceso a las aplicaciones de Streamlit.

- Las aplicaciones de Streamlit se ejecutan en almacenes de Snowflake y usan fases internas para almacenar archivos y datos.


### Visualice datos con Streamlit 🥋

Una de las excelentes ventajas de Streamlit es la facilidad con la que le permite interactuar con datos y visualizarlos. Ya las ha usado en las secciones **Cuestionario** al final de cada laboratorio de este curso. Con tan solo algunas líneas de código Python, puede crear cuadros y gráficos limpios y atractivos que dan vida a sus datos.

Ejecute la siguiente celda de Python para producir un gráfico de barras de Streamlit que visualice los resultados de la consulta analítica que acaba de ejecutar. Observe cómo se necesitan unas pocas líneas de código para crear este gráfico.

- ¿Qué plantas no tienen artículos en la base de conocimientos?

- ¿Qué plantas tienen más?

💡 **Consejo**: Como aprendió antes en este curso, puede hacer referencia a los resultados de las celdas anteriores en una celda posterior de un cuaderno de Snowflake. El siguiente código usa esta capacidad, según el resultado de la celda de consulta SQL analítica (**knowledge_base_analytical_query**) que acaba de ejecutar, para crear un DataFrame **de pandas**, que luego se analiza en Streamlit.


In [ ]:
import streamlit as st
import pandas as pd

chart_data = knowledge_base_analytical_query.to_pandas() # UTILIZA LA SALIDA DE UNA CELDA SQL ANTERIOR !!!

st.header("Artículos de la base de conocimientos por planta")
st.bar_chart(chart_data, x="PLANT_NAME", y="KB_ARTICLE_COUNT", color=['#33C4FF'])

## Hay mucho más que debe conocer de las funciones de IA y LLM de Snowflake 📓

Hay mucho más por conocer sobre las capacidades de la IA de Snowflake (y ML) y mucho más que podríamos abordar en este curso. 

No obstante, algunas de las funciones de Cortex LLM específicas de las tareas que son útiles son `TRANSLATE()` y `SUMMARIZE()`. Echemos un vistazo a éstas:

- [TRANSLATE()](https://docs.snowflake.com/en/sql-reference/functions/translate-snowflake-cortex): traduce el texto de entrada elegido de un idioma compatible a otro.

- [SUMMARIZE()](https://docs.snowflake.com/en/sql-reference/functions/summarize-snowflake-cortex): resume el texto de entrada en inglés elegido.

Echemos un vistazo a algunos ejemplos rápidos que usan los datos de nuestra base de conocimientos.


### Ejecute un ejemplo de `TRANSLATE()` 🥋 

Como vio en los ejemplos anteriores, acceder a las funciones de Cortex LLM usando SQL es simple. Podemos usar `TRANSLATE()` como lo haríamos con cualquier otra función SQL si se proporcionan las siguientes entradas:

- Una cadena que incluye el texto que se traducirá.

- Una cadena que especifica el código de idioma para el idioma en el que está actualmente el texto. Las opciones incluyen francés, alemán, italiano, japonés, coreano, español y más.

- Una cadena que especifica el código de idioma en el que se debe traducir el texto.

**Ejecute** el siguiente código de ejemplo a fin de traducir la información en la base de conocimientos para las plantas cuyo nombre comienza con `'C'` de **English** a **Spanish** y, luego, nuevamente a **English** de esa versión traducida.


In [ ]:
SELECT plant_name, 
       insight AS original_english_text,
       SNOWFLAKE.CORTEX.TRANSLATE(original_english_text, 'en', 'es') AS spanish_text,
       SNOWFLAKE.CORTEX.TRANSLATE(spanish_text, 'es', 'en') AS english_text_from_spanish,
FROM vegetable_knowledge_base
WHERE LEFT(plant_name,1) = 'C'; -- todos los nombres de plantas que comienzan con C

### Ejecute un ejemplo de `SUMMARIZE()` 🥋 

Como su nombre lo sugiere, la función `SUMMARIZE()` genera un resumen y este es del texto de entrada del idioma inglés dado. Se requiere solo un parámetro y esta es una cadena de texto que quiere resumir. 

**Ejecute** el siguiente ejemplo, que se centra en una planta individual de nuestra colección, la **Calabaza**:

- Usamos [LISTAGG](https://docs.snowflake.com/en/sql-reference/functions/listagg) para reunir todos los “factoids” (información) que tenemos para esta planta como un único cuerpo de texto.

- `SUMMARIZE()` se ejecuta en este cuerpo individual de texto, a partir del cual se genera un resumen.

- Empleamos una función de ayuda de Cortex LLM [COUNT_TOKENS()](https://docs.snowflake.com/en/sql-reference/functions/count_tokens-snowflake-cortex) para indicar el tamaño relativo del texto **antes** y **después** del resumen, pero puede, por supuesto, revisar los resultados por su cuenta.

💡 **Consejo**: Un token es la unidad más pequeña de texto procesada por las funciones de Cortex LLM de Snowflake, aproximadamente igual a cuatro caracteres. La equivalencia del texto de entrada o de salida sin procesar a los tokens puede variar según el modelo.

Continúe y ejecute el siguiente código, y revise el resultado:


In [ ]:
SELECT listagg(insight) AS all_insights,
        SNOWFLAKE.CORTEX.count_tokens('summarize', all_insights) AS all_insights_tokens,
        SNOWFLAKE.CORTEX.SUMMARIZE(listagg(insight)) AS summary,
        SNOWFLAKE.CORTEX.count_tokens('summarize', summary) AS summary_tokens
FROM vegetable_knowledge_base
WHERE plant_name = 'Pumpkin';

## Último, pero no menos importante, `COMPLETE()` 🥋

La función más sofisticada de Cortex LLM es [COMPLETE.](https://docs.snowflake.com/en/sql-reference/functions/complete-snowflake-cortex) En su forma más simple, la función toma una **solicitud** (texto e instrucciones sobre lo que quisiéramos que haga con ese texto) y genera una **respuesta** (completo) mediante su elección del modelo de lenguaje admitido.

Para tener una idea de cómo funciona esto, eche un vistazo al siguiente ejemplo:

- Aquí usamos el modelo **snowflake-arctic**.

- Proporcionamos instrucciones al modelo para ayudar a moldear su respuesta (por ejemplo, “Usted es un experto en TI”).

- Formulamos una pregunta (por ejemplo, “Explica qué es Snowflake”).

- La respuesta del modelo estará basada en sus conocimientos **innatos**, es decir, los datos que se usaron para entrenarlo, sin la información adicional que proporcionamos.

Ejecute el siguiente código y revise el resultado.


In [ ]:
SELECT SNOWFLAKE.CORTEX.COMPLETE('mixtral-8x7b', 'Usted es un experto en TI. Explica qué es Snowflake.') AS complete_response;

### Use los datos de nuestra base de conocimientos con `COMPLETE()` 🥋

Para nuestro ejemplo final, agreguemos **TODA** la información sobre jardines y plantas que tenemos en nuestra base de conocimientos y formulemos preguntas que queramos `COMPLETE()` para las cuales queremos generar respuestas, en función de esta información. 

### Explicación de la consulta `COMPLETE()`

La siguiente consulta puede parecer un poco intimidante, de modo que desglosemos en partes y expliquemos lo que sucede:

#### Sección 1

- La primera sección contiene preguntas de muestra para que las ejecute.

- Quite los comentarios cada vez que ejecute esta celda SQL para formular una nueva pregunta.

- Tenga en cuenta que estas consultas pueden tardar un poco en ejecutarse; tenga paciencia.

![Sección 1 de consulta completa (imagen)](https://edu-cdev-images.s3.us-west-2.amazonaws.com/ob/ob_complete_query_1.png)

#### Sección 2

- En la segunda sección, definimos la solicitud, que son nuestras instrucciones y nuestra pregunta del modelo al que se accede a través de la función `COMPLETE()`.

- Tenga en cuenta que incluimos etiquetas en la solicitud para identificar con claridad diferentes bloques de texto en la solicitud para el “consumo” del modelo.

![Sección 2 de consulta completa (imagen)](https://edu-cdev-images.s3.us-west-2.amazonaws.com/ob/ob_complete_query_2.png)

#### Sección 3

- La tercera sección es la llamada real a la función de Cortex.

- Tenga en cuenta que utilizamos las variables definidas anteriormente en la celda, los que flexibiliza su interacción.

![Sección 3 de consulta completa (imagen)](https://edu-cdev-images.s3.us-west-2.amazonaws.com/ob/ob_complete_query_3.png)


In [ ]:
-- preguntas de ejemplo
SET my_question  = 'which plant would take the shortest amount of time to cook';
--SET mi_pregunta = '¿qué plantas son mejores para jardineros principiantes';


-- la solicitud
SET prompt = 'You are a helpful gardening expert. Use only the supplied information <information> to answer the question posed <question>' || 
              $my_question || '</question> ' ||                 
             ' If you have no supplied information do not answer'; 
                
-- la llamada a la función Cortex
SELECT SNOWFLAKE.CORTEX.COMPLETE(
    'mixtral-8x7b', --ventana de contexto de 32K
    $my_question || 
    '<information>' ||
    (SELECT LISTAGG(insight, ' ') FROM vegetable_knowledge_base) ||
    '</information>'
) AS cortex_output;

## Cuestionario final :mag_right:

## Ponga a prueba sus conocimientos :mag_right:

Compruebe sus conocimientos con las preguntas interactivas que aparecen a continuación. Cada celda `RUN_THIS_QUIZ_QUESTION_` tiene un widget Streamlit que presenta una pregunta de opción múltiple sobre la funcionalidad de Snowflake.  

**Instrucciones:**  
1. Pase el cursor sobre la celda del cuaderno para revelar controles adicionales.
1. Haga clic en el botón ▶️ **Reproducir** en el lado derecho de cada celda del cuestionario para ejecutarla.  
1. Seleccione su respuesta entre las opciones que se proporcionan.  
1. Revise los comentarios antes de continuar.  

💡 **Nota:** Puede expandir la celda para ver el código si tiene curiosidad, pero no es obligatorio. Estos cuestionarios no son obligatorios: están disponibles para desafiar su transferencia de conocimientos y proporcionarle la oportunidad de practicar.  

:point_right: Estas preguntas pueden ser sobre cualquier contenido que haya abordado en este curso.


In [ ]:
st.divider()
question = "Si ejecutas una instrucción SELECT que ha funcionado en el pasado, pero ahora obtienes un mensaje de error que dice que la tabla no existe, ¿Qué debes verificar?"
options = ["Seleccione una opción a continuación...",
           "A) Tu proveedor de nube", 
           "B) Tu región actual", 
           "C) Tu rol de contexto",
           "D) Tu almacén de contexto",
           "E) El rol debajo de tu nombre en la esquina inferior izquierda"]

user_answer = st.radio(question, options, index=0)
if user_answer:
    if user_answer == "Seleccione una opción a continuación...":
        ''
    else:
        answer = 'd7c5064c385e4edcb1475f9c64f27e2e'
        # Extraer la letra de la opción (A, B, C o D)
        selected_option = user_answer.split(')')[0] + ')'
        response = session.sql(f"call common_db.resources.quiz_temp('{answer}', '{user_answer}', 'False')").collect()
        if response:
            value = response[0]['QUIZ_TEMP']
        st.write(f"{selected_option} {value}")

In [ ]:
st.divider()
question = "¿Cuál de estos objetos tiene la mayor potencia de cómputo?"
options = ["Seleccione una opción a continuación...",
           "A) Una base de datos de Snowflake", 
           "B) Un esquema de Snowflake", 
           "C) Una secuencia de Snowflake",
           "D) Un almacén de Snowflake",
           "E) Un almacén de datos de Snowflake"]

user_answer = st.radio(question, options, index=0)
if user_answer:
    if user_answer == "Seleccionme una opción a continuación...":
        ''
    else:
        answer = '4bd1cac6b25bd13acedc45a8d573370f'
        # Extraer la letra de la opción (A, B, C o D)
        selected_option = user_answer.split(')')[0] + ')'
        response = session.sql(f"call common_db.resources.quiz_temp('{answer}', '{user_answer}', 'False')").collect()
        if response:
            value = response[0]['QUIZ_TEMP']
        st.write(f"{selected_option} {value}")

In [ ]:
st.divider()
question = "¿Cuál de estos objetos es el más bajo en la jerarquía de la capa de almacenamiento?"
options = ["Seleccione una opción a continuación...",
           "A) Una base de datos de Snowflake", 
           "B) Un esquema de Snowflake", 
           "C) Una región de Snowflake",
           "D) Una cuenta de Snowflake",
           "E) Una tabla de Snowflake"]

user_answer = st.radio(question, options, index=0)
if user_answer:
    if user_answer == "Seleccione una opción a continuación...":
        ''
    else:
        answer = 'cd38f97f97287e1054f6b84d25274d79'
        # Extraer la letra de la opción (A, B, C o D)
        selected_option = user_answer.split(')')[0] + ')'
        response = session.sql(f"call common_db.resources.quiz_temp('{answer}', '{user_answer}', 'False')").collect()
        if response:
            value = response[0]['QUIZ_TEMP']
        st.write(f"{selected_option} {value}")

In [ ]:
st.divider()
question = "¿Cuál de las siguientes afirmaciones describe una fase(stage) externa en Snowflake?"
options = ["Seleccione una opción a continuación...",
           "A) Una fase se puede utilizar como un contador para generar identificadores únicos para cada nueva fila en una tabla. Asignamos un valor inicial y un incremento", 
           "B) Una fase nos permite agrupar tablas de base de datos. Para esto, creamos tres fases en la base de datos GARDEN_PLANTS. Una se llamaba VEGGIES", 
           "C) Una fase puede proporcionar una \"ventana\" entre Snowflake y una carpeta en la nube"]                

user_answer = st.radio(question, options, index=0)
if user_answer:
    if user_answer == "Seleccione una opción a continuación...":
        ''
    else:
        answer = 'f62cfda1b9028cc509b930b9f529af11'
        # Extraer la letra de la opción (A, B, C o D)
        selected_option = user_answer.split(')')[0] + ')'
        response = session.sql(f"call common_db.resources.quiz_temp('{answer}', '{user_answer}', 'False')").collect()
        if response:
            value = response[0]['QUIZ_TEMP']
        st.write(f"{selected_option} {value}")

In [ ]:
st.divider()
question = "¿Cuál de las siguientes no es una función LLM de Snowflake Cortex?"
options = ["Seleccione una opción a continuación...",
           "A) TRANSLATE", 
           "B) SUMMARIZE", 
           "C) PARSE_DOCUMENT", 
           "D) COUNT_TOKENS", 
           "E) CONTENT", 
           "F) COMPLETE"]                

user_answer = st.radio(question, options, index=0)
if user_answer:
    if user_answer == "Seleccione una opción a continuación...":
        ''
    else:
        answer = 'b71018c50689d88fc0f7aac8d3cbad47'
        # Extraer la letra de la opción (A, B, C o D)
        selected_option = user_answer.split(')')[0] + ')'
        response = session.sql(f"call common_db.resources.quiz_temp('{answer}', '{user_answer}', 'False')").collect()
        if response:
            value = response[0]['QUIZ_TEMP']
        st.write(f"{selected_option} {value}")

## Felicitaciones :tada: :confetti_ball:

Ha completado este curso: ¡buen trabajo!

